# 🎓 Course Recommender & Career Roadmap — ISI Group 2
### Complete, ready-to-run notebook · Modules **M0 → M7** + the **Module 8** web app

**How to run:** in the menu click **Runtime → Run all**. It takes ~3–4 minutes (the first
run downloads a small language model). When it finishes, the **last cell prints a public
link ending in `.gradio.live`** — open it, or share it, to use the live web app.

**What it does:** loads 623 real Coursera courses, works out which courses close your
**skill gap** for a target role, compares **two** recommender methods (keyword vs meaning),
scores them with **precision@k**, and writes a **career roadmap** with a free LLM.

**Career-roadmap key (optional):** the roadmap uses a free **Groq** key. No key? The
notebook still runs and shows a built-in sample roadmap — see **M7** for the 30-second
Colab Secrets setup if you want a live one.

## M0 · Setup

In [ ]:
# Install the three libraries Colab doesn't ship with (a few seconds).
!pip install -q gradio sentence-transformers groq

# Colab already has pandas, numpy, scikit-learn and matplotlib.
import pandas as pd, numpy as np, re, os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
print("setup ok")

## M1 · Load the data + the role → skills table
The course data is **fetched automatically** from the project's public GitHub repo (the
**Coursera Courses & Skills Dataset 2024**, 623 courses) — nothing to upload.

`role_skills` is our **hand-built** table of ~10 roles and the skills each one needs
(the main path, not a fallback).

In [ ]:
# Auto-download the dataset from the project repo (no upload needed).
DATA_URL = "https://raw.githubusercontent.com/devdutta/isi-2026-recommender/main/coursera_course_dataset_v3.csv"
raw = pd.read_csv(DATA_URL)
raw = raw.rename(columns={"Title":"title", "course_description":"description",
                          "Skills":"skills", "Organization":"provider",
                          "course_url":"url", "Difficulty":"level"})
print("columns ->", raw.columns.tolist())

role_skills = {
    "Data Scientist":  ["python","statistics","machine learning","sql","data visualization","deep learning"],
    "Data Analyst":    ["excel","sql","statistics","data visualization","python"],
    "ML Engineer":     ["python","machine learning","deep learning","sql","cloud"],
    "Data Engineer":   ["python","sql","data wrangling","cloud","big data","apis"],
    "Business Analyst":["excel","sql","statistics","data visualization","communication"],
    "BI Developer":    ["sql","data visualization","excel","statistics","power bi"],
    "AI Researcher":   ["python","deep learning","machine learning","mathematics","nlp"],
    "Backend Developer":["python","sql","apis","cloud","git"],
    "MLOps Engineer":  ["python","machine learning","cloud","docker","mlops"],
    "NLP Engineer":    ["python","machine learning","deep learning","nlp","statistics"],
}
print(raw.shape[0], "courses;", len(role_skills), "roles")

## M2 · Clean & prepare
We build the search text `text` from **Title + Skills** (both clean and fully populated).
We deliberately do **not** use the description column — only ~65% of rows have one, and
some are off-topic.

In [ ]:
courses = raw.copy()
courses = courses.dropna(subset=["title","skills"]).drop_duplicates("title").reset_index(drop=True)
courses["course_id"] = courses.index

def clean(s):
    s = re.sub(r"[^a-z0-9 ]", " ", str(s).lower())
    return re.sub(r"\s+", " ", s).strip()

courses["text"] = (courses["title"] + " " + courses["skills"]).apply(clean)
courses["skills"] = courses["skills"].fillna("").apply(
    lambda s: [x.strip().lower() for x in str(s).split(",") if x.strip()])
print("rows after clean:", len(courses))
courses[["course_id","title","skills","text"]].head(3)

## M3 · Your input → the skill gap
Change `current_skills` and `target_role` to try different learners.

In [ ]:
current_skills = ["python"]        # <- skills you already have
target_role    = "Data Scientist"  # <- pick any key from role_skills
assert target_role in role_skills, f"Pick a role from {list(role_skills)}"

have = {s.lower() for s in current_skills}
skill_gap = [s for s in role_skills[target_role] if s.lower() not in have]
query_text = " ".join(skill_gap)
print("skill_gap:", skill_gap)

## M4 · TF-IDF recommender (keyword baseline)

In [ ]:
vec    = TfidfVectorizer(stop_words="english")
matrix = vec.fit_transform(courses["text"])
q_vec  = vec.transform([query_text])
scores = cosine_similarity(q_vec, matrix).flatten()
K   = 10
top = scores.argsort()[::-1][:K]
tfidf_recs = courses.loc[top, ["course_id","title"]].copy()
tfidf_recs["score"] = scores[top]; tfidf_recs["rank"] = range(1, len(tfidf_recs)+1)
tfidf_recs

## M5 · Embedding recommender (semantic — matches *meaning*)
The first run downloads a small sentence-embedding model (~40s), then encodes all 623 courses.

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")
course_emb = model.encode(courses["text"].tolist(), show_progress_bar=False)
q_emb      = model.encode([query_text])
scores_e   = cosine_similarity(q_emb, course_emb).flatten()
top_e = scores_e.argsort()[::-1][:K]
embed_recs = courses.loc[top_e, ["course_id","title"]].copy()
embed_recs["score"] = scores_e[top_e]; embed_recs["rank"] = range(1, len(embed_recs)+1)
embed_recs

## M6 · Evaluate & compare (the graded core)
`is_relevant()` uses **whole-word matching** so a gap token like `statistics` matches the
dataset phrase `statistics for data science` (an exact set-intersection would miss it).

In [ ]:
def is_relevant(course_skills, skill_gap):
    skills_text = " ".join(course_skills).lower()
    return any(re.search(r"\b" + re.escape(g) + r"\b", skills_text) for g in skill_gap)

def precision_at_k(recs, courses, skill_gap, k=10):
    hits = 0
    for cid in recs.head(k)["course_id"]:
        cskills = courses.loc[courses["course_id"]==cid, "skills"].iloc[0]
        hits += is_relevant(cskills, skill_gap)
    return hits / k

p_tfidf = precision_at_k(tfidf_recs, courses, skill_gap, k=K)
p_embed = precision_at_k(embed_recs, courses, skill_gap, k=K)
eval_table = pd.DataFrame({"method":["TF-IDF","Embedding"], f"precision@{K}":[p_tfidf, p_embed]})
# TIE RULE: >= means a tie (1.0 == 1.0) goes to Embedding. Use > if you'd rather it go to TF-IDF.
winner = "Embedding" if p_embed >= p_tfidf else "TF-IDF"
print(eval_table.to_string(index=False)); print("winner:", winner)
# A single user at k=10 can tie at 1.0 - evaluate several roles & k values for real discrimination.
eval_table.plot.bar(x="method", y=f"precision@{K}", legend=False)
plt.title("Which recommender is better?"); plt.ylabel(f"precision@{K}"); plt.ylim(0,1.05); plt.show()

## M7 · Career roadmap (free Groq LLM — optional)
This writes a month-by-month roadmap over the recommended courses.

**To use a live roadmap (optional, 30 seconds):**
1. Get a free key at **console.groq.com** (no credit card).
2. In Colab, click the **🔑 key icon** in the left sidebar (**Secrets**).
3. **+ Add new secret** → Name: **`GROQ_API_KEY`**, Value: *your key*.
4. Turn **ON** the **Notebook access** toggle for it.
5. Re-run the notebook.

**Skip it?** Fine — the notebook shows a built-in sample roadmap and never breaks.

In [ ]:
# Pull the Groq key from Colab Secrets (the secure home for keys). If it isn't set,
# we quietly fall back to an offline sample roadmap so the notebook still runs.
try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY").strip()  # .strip() drops any stray newline
    print("Groq key found in Colab Secrets.")
except Exception:
    print("No Groq key in Secrets - roadmap will use the offline sample (this is fine).")

In [ ]:
def make_roadmap(current_skills, target_role, skill_gap, best_recs):
    # Uses Groq if GROQ_API_KEY is set; otherwise returns a simple offline list.
    course_list = "\n".join(f"- {t}" for t in best_recs["title"].head(8))
    api_key = os.environ.get("GROQ_API_KEY")
    if api_key:
        try:
            from groq import Groq
            prompt = (
                f"You are helping a learner plan their upskilling.\n\n"
                f"Current skills: {', '.join(current_skills) or 'none'}\n"
                f"Target role: {target_role}\n"
                f"Skills they still need (the gap): {', '.join(skill_gap)}\n\n"
                f"Recommended courses - use ONLY these, do not invent any:\n{course_list}\n\n"
                "Write a month-by-month learning roadmap that uses ONLY the courses above, "
                "orders foundational skills before advanced ones, one short sentence per course, "
                "realistic over ~6-8 months, ending with a line that this is a SUGGESTED order, "
                "not a verified prerequisite plan."
            )
            resp = Groq(api_key=api_key).chat.completions.create(
                model="openai/gpt-oss-120b",
                messages=[{"role": "user", "content": prompt}])
            return resp.choices[0].message.content, True
        except Exception as e:
            print("Groq call failed, using offline sample:", e)
    lines = [f"Suggested roadmap toward {target_role} (offline sample):", ""]
    lines += [f"- Month {i}: {t}" for i, t in enumerate(best_recs["title"].head(8), 1)]
    lines += ["", "This is a SUGGESTED order, not a verified prerequisite plan."]
    return "\n".join(lines), False

best_recs = embed_recs if winner == "Embedding" else tfidf_recs
roadmap_text, used_llm = make_roadmap(current_skills, target_role, skill_gap, best_recs)
print(("LIVE (Groq)" if used_llm else "OFFLINE sample") + " roadmap:\n")
print(roadmap_text)

## 🖥 Module 8 · The web app  ← this cell gives you the shareable link
Run this cell. It wraps everything above behind a web form and, thanks to `share=True`,
prints a **public `.gradio.live` link**. Open it to use the tool; share it to show it off.
The link stays live while this notebook keeps running.

In [ ]:
import gradio as gr

def _recommend(scores, k):
    top = scores.argsort()[::-1][:k]
    recs = courses.loc[top, ["course_id", "title"]].copy()
    recs["score"] = scores[top].astype(float).round(3)
    recs["rank"] = range(1, len(recs) + 1)
    return recs[["rank", "course_id", "title", "score"]].reset_index(drop=True)

def run(target_role, current_skills, k):
    have = {s.lower() for s in (current_skills or [])}
    gap = [s for s in role_skills[target_role] if s.lower() not in have]              # M3
    if not gap:
        note = f"You already have every skill {target_role} needs. Try another role."
        empty = pd.DataFrame(columns=["rank","course_id","title","score"])
        return note, empty, empty, "", ""
    q = " ".join(gap)
    tf = _recommend(cosine_similarity(vec.transform([q]), matrix).flatten(), k)        # M4
    em = _recommend(cosine_similarity(model.encode([q]), course_emb).flatten(), k)     # M5
    p_tf = precision_at_k(tf, courses, gap, k)                                          # M6
    p_em = precision_at_k(em, courses, gap, k)                                          # M6
    winner = "Embedding" if p_em >= p_tf else "TF-IDF"
    best = em if winner == "Embedding" else tf
    gap_md = f"**Skill gap:** {', '.join(gap)}"
    if abs(p_tf - p_em) < 1e-9:
        win_md = (f"**Tie** - both score precision@{k} = {p_tf:.2f}. The methods differ in "
                  "*which* courses they surface (keywords vs meaning).")
    else:
        win_md = f"**Winner: {winner}** - precision@{k}: TF-IDF {p_tf:.2f} vs Embedding {p_em:.2f}"
    roadmap, _ = make_roadmap(list(have), target_role, gap, best)                       # M7
    roadmap = "  \n".join(line for line in roadmap.splitlines() if line.strip())
    return gap_md, tf, em, win_md, roadmap

# Force light mode so everyone sees the same UI regardless of their browser's dark setting.
_css = """
:root, html.dark, .dark {
  color-scheme: light !important;
  --bg-dark: #f8fafc !important; --col-dark: #1e293b !important;
  --body-background-fill: #f8fafc !important; --body-text-color: #1e293b !important;
  --background-fill-primary: #ffffff !important; --background-fill-secondary: #f1f5f9 !important;
  --block-background-fill: #eef2fb !important; --input-background-fill: #ffffff !important;
  --border-color-primary: #dbe3f2 !important; --input-border-color: #cdd7ea !important;
  --table-even-background-fill: #ffffff !important; --table-odd-background-fill: #f4f7fd !important;
  --neutral-900: #eef2fb !important; --neutral-950: #ffffff !important;
}
html, body, .gradio-container { background: #ffffff !important; color: #1e293b !important; }
.block { box-shadow: 0 1px 2px rgba(15,23,42,.05) !important; }
.token { background: #dbe4ff !important; color: #312e81 !important;
         border: 1px solid #c3cffb !important; border-radius: 8px !important; }
.token .token-remove, .token-remove { color: #4f46e5 !important; background: transparent !important; }
"""
with gr.Blocks(title="Course Recommender", css=_css) as demo:
    gr.Markdown("# 🎓 Course Recommender & Career Roadmap\nPick a role and the skills you already have.")
    with gr.Row():
        role_in = gr.Dropdown(list(role_skills), value="Data Scientist", label="Target role")
        skills_in = gr.Dropdown(sorted({s for v in role_skills.values() for s in v}),
                                value=["python"], multiselect=True, label="Skills you already have")
        k_in = gr.Slider(5, 15, value=10, step=1, label="Top-k")
    go = gr.Button("Get recommendations & roadmap", variant="primary")
    gap_out = gr.Markdown()
    with gr.Row():
        tfidf_out = gr.Dataframe(label="TF-IDF (baseline)", interactive=False)
        embed_out = gr.Dataframe(label="Embedding (semantic)", interactive=False)
    winner_out = gr.Markdown()
    roadmap_out = gr.Markdown()
    go.click(run, [role_in, skills_in, k_in],
             [gap_out, tfidf_out, embed_out, winner_out, roadmap_out])

# share=True is what gives you the public link in Colab.
demo.launch(share=True)

## Honest notes (own these in the report)
- **precision@k uses a proxy** (`is_relevant()`): it calls a course "relevant" only if a
  gap word appears in its skill tags — so the two methods often **tie**. The real story is
  *which* courses each surfaces, not that one number beats the other.
- **The roadmap order is *suggested*, not verified** — the LLM guesses a sensible order; it
  does not check true prerequisites.
- The recommender has **no concept of the role** — it is pure text similarity to the gap words.